# 07 - K-Means Clustering
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## Segmenting the Portfolio with K-Means
K-Means partitions data points into $k$ groups by locating cluster centroids and assigning each borrower to the nearest center.
In this step, we:
1. Use the **Elbow Method** to identify the best number of clusters ($k$).
2. Fit our K-Means model.
3. Map borrowers to their clusters and visualize them in our 2D PCA space.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_scaled = pd.read_csv("../data/processed/scaled_features.csv")
df = pd.read_csv("../data/processed/cleaned_loans_dataset.csv")

print(f"Ready to cluster {len(X_scaled):,} borrowers.")


---
## 1. Finding Optimal $k$ with the Elbow Method
We test cluster counts from 2 to 7 and track **Inertia** (within-cluster sum of squared distances). Where the curve bends noticeably marks our elbow point.


In [ ]:
inertias = []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker='s', color='teal', linewidth=2)
plt.title("The Elbow Method for Selecting k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.tight_layout()
plt.savefig("../outputs/figures/kmeans_elbow_curve.png", dpi=300)
plt.show()


---
## 2. Fitting K-Means with $k=4$
The curve flattens around $k=4$, balancing model simplicity with meaningful separation.


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

print("Borrower distribution by cluster:")
print(df['KMeans_Cluster'].value_counts().sort_index())


---
## 3. Visualizing Clusters in PCA Space


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='KMeans_Cluster', 
    data=df.sample(2500, random_state=42), 
    palette='Set1', 
    alpha=0.7
)
plt.title("Borrower Segments Identified by K-Means (in PCA Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")

plt.tight_layout()
plt.savefig("../outputs/figures/kmeans_clusters_pca.png", dpi=300)
plt.show()
